# **Ablation Studies**

In [ ]:
# ==========================================
# Ablation Study: Isolating ISR Components
# ==========================================
import os
import gc
import warnings
import math
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F_func
from sentence_transformers import SentenceTransformer
from beir import util
from beir.datasets.data_loader import GenericDataLoader

os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
warnings.filterwarnings("ignore")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"--- Running Ablations on: {DEVICE.type.upper()} ---")

# --- Architecture ---
class ISR_Bottleneck(nn.Module):
    def __init__(self, dense_dim=768, sparse_dim=4096):
        super(ISR_Bottleneck, self).__init__()
        self.encoder = nn.Linear(dense_dim, sparse_dim)
        self.relu = nn.ReLU()
        self.decoder = nn.Linear(sparse_dim, dense_dim)

    def forward(self, dense_embeds):
        F = self.relu(self.encoder(dense_embeds))
        return F, self.decoder(F)

def train_isr(model, train_embeddings, epochs=150, l1_lambda=1.5e-6, ip_lambda=3.0):
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    mse_loss_fn = nn.MSELoss()
    model.train()
    for _ in range(epochs):
        optimizer.zero_grad()
        F, rec_E = model(train_embeddings)

        # 1. Reconstruction Loss
        rec_loss = mse_loss_fn(rec_E, train_embeddings)
        # 2. Sparsity Loss
        sparse_loss = l1_lambda * torch.norm(F, p=1)

        # 3. Contrastive/IP Loss
        if ip_lambda > 0:
            F_norm = F_func.normalize(F, p=2, dim=1, eps=1e-8)
            sim_F = torch.matmul(F_norm, F_norm.T)
            sim_E = torch.matmul(train_embeddings, train_embeddings.T)
            ip_loss = mse_loss_fn(sim_F, sim_E)
        else:
            ip_loss = 0.0

        total_loss = rec_loss + sparse_loss + (ip_lambda * ip_loss)
        total_loss.backward()
        optimizer.step()
    return model

def compute_ndcg_at_k(retrieved_idx, relevant_idx, k=10):
    dcg = sum(1.0 / math.log2(i + 2) for i, idx in enumerate(retrieved_idx[:k]) if idx in relevant_idx)
    idcg = sum(1.0 / math.log2(i + 2) for i in range(min(len(relevant_idx), k)))
    return dcg / idcg if idcg > 0 else 0.0

# --- Execution ---
def run_ablations():
    # Setup Data (SciDocs Only for speed)
    url = "https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/scidocs.zip"
    out_dir = util.download_and_unzip(url, "datasets")
    corpus, queries, qrels = GenericDataLoader(data_folder=out_dir).load(split="test")

    query_ids = list(queries.keys())[:20]
    relevant_doc_ids = []
    for qid in query_ids:
        if qid in qrels: relevant_doc_ids.extend([did for did, score in qrels[qid].items() if score > 0])

    all_doc_ids = list(set(relevant_doc_ids + list(corpus.keys())[:500]))
    doc_id_to_idx = {did: i for i, did in enumerate(all_doc_ids)}

    sample_docs = [corpus[did].get("text") for did in all_doc_ids]
    sample_queries = [queries[qid] for qid in query_ids]

    # Embed Base Vectors
    print("Encoding Base Vectors...")
    base_model = SentenceTransformer('facebook/contriever', device=DEVICE)
    d_dense = F_func.normalize(torch.tensor(base_model.encode(sample_docs, batch_size=64)).to(DEVICE), p=2, dim=1)
    q_dense = F_func.normalize(torch.tensor(base_model.encode(sample_queries, batch_size=64)).to(DEVICE), p=2, dim=1)
    train_embeddings = torch.cat([d_dense, q_dense], dim=0)

    # --- Ablation Configurations ---
    configs = [
        {"name": "Optimal ISR", "k": 4096, "l1": 1.5e-6, "ip": 3.0},
        {"name": "Expanded Dim", "k": 8192, "l1": 1.5e-6, "ip": 3.0},
        {"name": "No IP/Contrastive", "k": 4096, "l1": 1.5e-6, "ip": 0.0},
        {"name": "No Sparsity (L1=0)", "k": 4096, "l1": 0.0, "ip": 3.0}
    ]

    results = []

    for cfg in configs:
        print(f"Training: {cfg['name']} (k={cfg['k']}, L1={cfg['l1']}, IP={cfg['ip']})...")
        model = ISR_Bottleneck(dense_dim=768, sparse_dim=cfg['k']).to(DEVICE)
        model = train_isr(model, train_embeddings, epochs=150, l1_lambda=cfg['l1'], ip_lambda=cfg['ip'])
        model.eval()

        with torch.no_grad():
            q_F, _ = model(q_dense)
            d_F, _ = model(d_dense)

        q_F_norm = F_func.normalize(q_F, p=2, dim=1, eps=1e-8)
        d_F_norm = F_func.normalize(d_F, p=2, dim=1, eps=1e-8)

        scores = torch.matmul(q_F_norm, d_F_norm.T)
        ndcg_list, dts_list = [], []

        for i, qid in enumerate(query_ids):
            if qid not in qrels: continue
            relevant_idx = [doc_id_to_idx[did] for did in qrels[qid].keys() if did in doc_id_to_idx]
            if not relevant_idx: continue

            top_k_idx = torch.topk(scores[i], k=10).indices.tolist()
            ndcg_list.append(compute_ndcg_at_k(top_k_idx, relevant_idx, k=10))

            q_facet = torch.argmax(q_F_norm[i]).item()
            dts_matches = sum(1 for idx in top_k_idx if q_facet in torch.topk(d_F_norm[idx], k=3).indices.tolist())
            dts_list.append((dts_matches / 10.0) * 100)

        sparsity = (1.0 - ((d_F_norm > 0).float().sum(dim=1).mean().item() / cfg['k'])) * 100

        results.append({
            "name": cfg['name'],
            "ndcg": np.mean(ndcg_list),
            "dts": np.mean(dts_list),
            "sparsity": sparsity
        })

    # --- Output Table ---
    print("\n\n=========================================================================")
    print(" Table 2: Ablation Study on ISR Components (SciDocs)")
    print("=========================================================================")
    print(f"{'Configuration':<25} | {'NDCG@10':<9} | {'DTS (%)':<9} | {'Zero-Vals (%)'}")
    print("-" * 65)
    for r in results:
        print(f"{r['name']:<25} | {r['ndcg']:<9.4f} | {r['dts']:<9.1f} | {r['sparsity']:.1f}%")
    print("=========================================================================")

if __name__ == "__main__":
    run_ablations()

--- Running Ablations on: CUDA ---


  0%|          | 0/25657 [00:00<?, ?it/s]

Encoding Base Vectors...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Training: Optimal ISR (k=4096, L1=1.5e-06, IP=3.0)...
Training: Expanded Dim (k=8192, L1=1.5e-06, IP=3.0)...
Training: No IP/Contrastive (k=4096, L1=1.5e-06, IP=0.0)...
Training: No Sparsity (L1=0) (k=4096, L1=0.0, IP=3.0)...


 Table 2: Ablation Study on ISR Components (SciDocs)
Configuration             | NDCG@10   | DTS (%)   | Zero-Vals (%)
-----------------------------------------------------------------
Optimal ISR               | 0.4177    | 85.5      | 97.4%
Expanded Dim              | 0.3968    | 89.5      | 98.8%
No IP/Contrastive         | 0.0500    | 100.0     | 100.0%
No Sparsity (L1=0)        | 0.3946    | 6.0       | 82.0%
